# Multi-Repository Artifact Classification

This notebook classifies AI coding tool artifacts from **multiple repositories** into predefined semantic categories using **Nearest Centroid Classification** on embedding vectors.

**Approach:** We use 9 predefined category templates as fixed centroids and classify each artifact by cosine similarity.

**Input:** Pre-filtered data from `2. embedding_filtration.ipynb` (stored in `data/` directory).

**Output:** Classification results with UMAP visualizations and quality analysis.

## Setup

In [ ]:
# IMPORTANT: Set OpenMP environment variables BEFORE any other imports to prevent kernel crash on macOS ARM
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Suppress duplicate OpenMP library conflicts
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"  # Replacement for deprecated omp_set_nested
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import pickle
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity

# Add src to path for imports
sys.path.insert(0, os.path.abspath('..'))

from src.embedding_generator import load_embedding_model

## Configuration

In [ ]:
# =============================================================================
# DATA SOURCE: Pre-filtered data from notebook 2 (embedding_filtration)
# =============================================================================
PROJECT_ROOT = Path('..')
DATA_DIR = PROJECT_ROOT / 'data'

# Choose which filtered dataset to use:
#   'common'   - All artifacts after common filters (empty, single-repo, readme, outlier, LOF)
#   'ai_tools' - Only repos containing at least one known AI artifact
DATASET = 'ai_tools'

DATASET_PATHS = {
    'common': {
        'embeddings': DATA_DIR / 'filtered_common_embeddings.npz',
        'metadata': DATA_DIR / 'filtered_common_metadata.csv',
    },
    'ai_tools': {
        'embeddings': DATA_DIR / 'filtered_ai_tools_embeddings.npz',
        'metadata': DATA_DIR / 'filtered_ai_tools_metadata.csv',
    },
}

print(f"Dataset: {DATASET}")
print(f"Embeddings: {DATASET_PATHS[DATASET]['embeddings']}")
print(f"Metadata: {DATASET_PATHS[DATASET]['metadata']}")

# Classification settings
CONFIDENCE_THRESHOLD = 0.25  # Below this, mark as "low_confidence"
AMBIGUITY_MARGIN = 0.05      # If top-2 categories within this margin, flag as ambiguous

# Embedding model (must match what was used in collection)
MODEL_NAME = 'nomic-ai/nomic-embed-text-v1.5'

# Category templates - semantic descriptions for each category
# These serve as FIXED CENTROIDS for nearest-neighbor classification
# Optimized for embedding-based matching: keyword-dense, no filler words
CATEGORY_TEMPLATES = {
    "agents": (
        "A persona definition file that establishes an AI agent's identity, role, and behavioral boundaries. "
        "Contains YAML frontmatter with structured fields like name, type, model, tools, and capabilities. "
        "Defines delegation boundaries, domain expertise scope, and interaction protocols for a single autonomous agent."
    ),

    "commands": (
        "A short, self-contained prompt template that defines exactly one executable action a user can invoke. "
        "Typically under 25 lines with a slash-command trigger, parameterized $ARGUMENTS, and a single output. "
        "Not a multi-step orchestration or policy document \u2014 just one atomic, reusable operation like commit, review, or format."
    ),

    "flows": (
        "A multi-phase orchestration plan that coordinates several agents or steps through a complex workflow. "
        "Contains tables with worker assignments, timeline phases, and dependency mapping between stages. "
        "Agents are spawned, monitored, and their outputs synthesized \u2014 this is a project execution blueprint, not a set of rules or policies."
    ),

    "rules": (
        "A policy document of imperative directives that govern how an AI assistant must behave in a codebase. "
        "Uses mandatory language like NEVER, ALWAYS, MUST, and DO NOT to enforce constraints and conventions. "
        "Does not contain code examples, workflow orchestration tables, or step-by-step tutorials \u2014 only behavioral rules and project-level instructions."
    ),

    "skills": (
        "A long-form how-to guide (typically 200-600 lines) that teaches a specific technique or capability in depth. "
        "Includes trigger conditions, detailed step-by-step methodology, MCP tool usage, edge case handling, and validation criteria. "
        "Functions as reusable domain expertise that can be composed and extended, unlike short commands or behavioral rules."
    ),

    "architecture": (
        "A system design document describing software architecture with component diagrams, data flows, and deployment topology. "
        "Uses Mermaid, PlantUML, or ASCII diagrams with ADR-style decision records and C4 model levels. "
        "Covers infrastructure, service boundaries, scaling strategies, and technology stack rationale \u2014 not coding standards or runtime configuration."
    ),

    "code-style": (
        "A coding standards document with before-and-after code comparisons showing incorrect vs correct patterns. "
        "Contains inline code examples, linting rules, type safety requirements, naming conventions with specific casing, and coverage metrics. "
        "Focuses on how source code should be written at the syntax level \u2014 unlike behavioral rules which govern AI assistant conduct."
    ),

    "configuration": (
        "A machine-readable JSON, YAML, or TOML file with hierarchical key-value pairs, boolean flags, and nested settings objects. "
        "Defines tool servers, environment variables, permission scopes, file patterns, and feature toggles. "
        "Contains no prose paragraphs or natural language instructions \u2014 purely structured data for tool or environment configuration."
    ),

    "session-logs": (
      "A retrospective record of work completed by an AI agent, capturing task outcomes, status transitions, and timestamped activity. "
      "Contains structured metadata like status, acceptance criteria checklists, files modified, commits made, or transition logs with actor attribution. "
      "Backward-looking and observational \u2014 documents what was done and when, unlike rules that prescribe behavior or flows that plan future work."
    )
}

print(f"Defined {len(CATEGORY_TEMPLATES)} categories: {list(CATEGORY_TEMPLATES.keys())}")

## Phase 1: Load Pre-Filtered Data

Load embeddings and metadata from the filtered datasets produced by `2. embedding_filtration.ipynb`.

In [ ]:
# Load pre-filtered data from notebook 2
paths = DATASET_PATHS[DATASET]

# Load embeddings
emb_data = np.load(paths['embeddings'], allow_pickle=True)
embeddings = emb_data['embeddings']
global_file_ids_from_npz = emb_data['global_file_ids']

# Load metadata
metadata_df = pd.read_csv(paths['metadata'])

print(f"Loaded dataset: {DATASET}")
print(f"  Embeddings shape: {embeddings.shape}")
print(f"  Metadata rows: {len(metadata_df)}")
print(f"  Metadata columns: {list(metadata_df.columns)}")

In [ ]:
# Verify alignment and set up variables used by downstream cells
assert len(embeddings) == len(metadata_df), (
    f"Mismatch: {len(embeddings)} embeddings vs {len(metadata_df)} metadata rows"
)

# Use global_file_id from metadata (authoritative after filtering)
if 'global_file_id' in metadata_df.columns:
    all_file_ids = metadata_df['global_file_id'].tolist()
else:
    all_file_ids = [f"file_{i:05d}" for i in range(len(metadata_df))]
    metadata_df['global_file_id'] = all_file_ids

all_repo_labels = metadata_df['repo_name'].tolist()
model_name = MODEL_NAME

print(f"\nData summary:")
print(f"  Total files: {len(all_file_ids):,}")
print(f"  Embeddings shape: {embeddings.shape}")
print(f"  Repositories: {metadata_df['repo_name'].nunique():,}")
print(f"  Model: {model_name}")
print(f"\nFiles per repository (top 20):")
for repo, count in pd.Series(all_repo_labels).value_counts().head(20).items():
    print(f"  - {repo}: {count}")
print(f"\n\u2713 Data loaded and aligned")

In [ ]:
# Quick data integrity check
print("Data integrity checks:")

# Check for NaN/zero embeddings (should have been filtered by notebook 2)
has_nan = np.any(np.isnan(embeddings), axis=1)
is_zero = np.all(embeddings == 0, axis=1)
print(f"  NaN embeddings: {has_nan.sum()} (expected 0)")
print(f"  Zero embeddings: {is_zero.sum()} (expected 0)")

# Check embedding dimensions
print(f"  Embedding dimensions: {embeddings.shape[1]}")
assert embeddings.shape[1] == 768, f"Expected 768-dim embeddings, got {embeddings.shape[1]}"

# Show artifact type distribution
print(f"\nArtifact types:")
for name, count in metadata_df['artifact_name'].value_counts().head(15).items():
    print(f"  {name}: {count:,}")

print(f"\n\u2713 All checks passed")

In [ ]:
# Calculate artifact counts per repository (from all_repo_labels)
repo_counts = pd.Series(all_repo_labels)

# Get distribution of artifacts per repository
artifact_counts = repo_counts.value_counts()

# Use custom bins to handle skewed distribution
# numpy histogram uses [left, right) intervals
max_artifacts = artifact_counts.values.max()
bins = [0, 1, 2, 3, 5, 10, 20, 50, 100, max(200, max_artifacts + 1)]
bin_labels = ['0', '1', '2', '3-4', '5-9', '10-19', '20-49', '50-99', '100+']

bin_counts, bin_edges = np.histogram(artifact_counts.values, bins=bins)
total_repos = len(artifact_counts)
percent_per_bin = bin_counts / total_repos * 100

import plotly.graph_objects as go

fig = go.Figure(data=[go.Bar(
    x=bin_labels,
    y=bin_counts,
    marker_color='steelblue',
    opacity=0.85,
    hovertext=[f"Repos: {count}<br>{pct:.1f}%" for count, pct in zip(bin_counts, percent_per_bin)],
    hoverinfo='text'
)])

# Add percent annotations on top of bars
for i, (yc, pc) in enumerate(zip(bin_counts, percent_per_bin)):
    if yc > 0:
        fig.add_annotation(
            x=i,
            y=yc,
            text=f"{pc:.1f}%",
            showarrow=False,
            yshift=12,
            font=dict(size=11, color="black")
        )

# Add headroom for annotations
y_max = max(bin_counts) * 1.12

fig.update_layout(
    title='Distribution of Number of Artifacts per Repository',
    xaxis_title='Number of Artifacts per Repository',
    yaxis_title='Number of Repositories (log scale)',
    yaxis=dict(type='log', range=[0, np.log10(y_max) + 0.1]),
    template='simple_white',
    height=500,
    bargap=0.15
)

fig.show()

# Also show summary statistics
print(f"\nSummary Statistics:")
print(f"  Total repositories: {total_repos:,}")
print(f"  Min artifacts: {artifact_counts.values.min()}")
print(f"  Max artifacts: {artifact_counts.values.max()}")
print(f"  Median artifacts: {np.median(artifact_counts.values):.0f}")
print(f"  Mean artifacts: {np.mean(artifact_counts.values):.1f}")

In [ ]:
# Group metadata_df by tool_name and count artifacts per tool across all repositories
tool_counts = metadata_df['tool_name'].value_counts().sort_values(ascending=False)

# Calculate percentage of repositories that have at least one artifact with each tool_name
# For each tool, get the unique repos that use that tool, then percent of total repos
repo_tool = metadata_df.groupby('tool_name')['repo_name'].nunique().sort_values(ascending=False)
total_repos = metadata_df['repo_name'].nunique()
repo_percent = (repo_tool / total_repos * 100).round(1)

import plotly.graph_objects as go

# Prepare hover text showing both artifact count and percent of repos
hover_text = [
    f"{tool}<br>Artifacts: {tool_counts[tool]}<br>Repos: {repo_tool[tool]} ({repo_percent[tool]}%)"
    for tool in tool_counts.index
]

fig = go.Figure(data=[go.Bar(
    x=tool_counts.index,
    y=tool_counts.values,
    marker=dict(color='skyblue'),
    text=[f"{repo_percent[t]}%" for t in tool_counts.index],
    textposition='outside',
    hovertext=hover_text,
    hoverinfo='text',
)])

fig.update_layout(
    title='Number of Artifacts per Tool (all repositories)',
    xaxis_title='Tool',
    yaxis_title='Number of Artifacts',
    xaxis_tickangle=-45,
    template='simple_white'
)

fig.show()


## Phase 2: Embed Category Templates

In [ ]:
# Load the same model used for artifact embeddings
print(f"Loading model: {model_name}")
model = load_embedding_model(model_name)

# Embed all category templates with the same task prefix used for artifact embeddings.
# nomic-embed-text-v1.5 requires a task prefix ("clustering: ") to place texts in the
# correct region of embedding space. Artifact embeddings were generated with this prefix
# via add_embeddings_to_artifacts(), so category centroids must match.
from src.embedding_generator import DEFAULT_TASK_PREFIX

TASK_PREFIX = DEFAULT_TASK_PREFIX  # "clustering"

print(f"\nEmbedding {len(CATEGORY_TEMPLATES)} category templates (prefix='{TASK_PREFIX}')...")
category_embeddings = {}
for category, template_text in CATEGORY_TEMPLATES.items():
    prefixed = f"{TASK_PREFIX}: {template_text}"
    category_embeddings[category] = model.encode(prefixed)
    print(f"  - {category}: embedded")

print("\nCategory embeddings complete!")

# Clean up model to free memory
del model
gc.collect()
print("Model unloaded to free memory.")

## Phase 3: Nearest Centroid Classification

Classify each file by computing cosine similarity to all category centroids and selecting the highest.

In [ ]:
# Build category embeddings matrix for vectorized similarity computation
category_names = list(category_embeddings.keys())
category_matrix = np.vstack([category_embeddings[cat] for cat in category_names])

print(f"Category matrix shape: {category_matrix.shape}")
print(f"File embeddings shape: {embeddings.shape}")

# Compute cosine similarity: each file vs all categories
# Result shape: (n_files, n_categories)
similarity_matrix = cosine_similarity(embeddings, category_matrix)

print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"\nSimilarity stats:")
print(f"  Min: {similarity_matrix.min():.3f}")
print(f"  Max: {similarity_matrix.max():.3f}")
print(f"  Mean: {similarity_matrix.mean():.3f}")

## Phase 4: Assign Categories with Confidence Scores

In [ ]:
# For each file, get top-2 categories with scores
classifications = []

for i in range(len(all_file_ids)):
    scores = similarity_matrix[i]
    
    # Sort indices by score (descending)
    sorted_indices = np.argsort(scores)[::-1]
    
    # Top 1
    top1_idx = sorted_indices[0]
    top1_category = category_names[top1_idx]
    top1_score = scores[top1_idx]
    
    # Top 2
    top2_idx = sorted_indices[1]
    top2_category = category_names[top2_idx]
    top2_score = scores[top2_idx]
    
    # Compute margin between top-1 and top-2
    margin = top1_score - top2_score
    
    # Flag ambiguous cases
    is_ambiguous = margin < AMBIGUITY_MARGIN
    is_low_confidence = top1_score < CONFIDENCE_THRESHOLD
    
    classifications.append({
        'global_file_id': all_file_ids[i],
        'repo_name': all_repo_labels[i],
        'category': top1_category,
        'confidence': top1_score,
        'second_category': top2_category,
        'second_confidence': top2_score,
        'margin': margin,
        'is_ambiguous': is_ambiguous,
        'is_low_confidence': is_low_confidence,
    })

# Convert to DataFrame
class_df = pd.DataFrame(classifications)

print(f"Classified {len(class_df)} files across {class_df['repo_name'].nunique()} repositories")
print(f"\nCategory distribution (all repos):")
print(class_df['category'].value_counts())

print(f"\nAmbiguous files (margin < {AMBIGUITY_MARGIN}): {class_df['is_ambiguous'].sum()}")
print(f"Low confidence files (score < {CONFIDENCE_THRESHOLD}): {class_df['is_low_confidence'].sum()}")

In [ ]:
# Category distribution by repository
print("=" * 80)
print("CATEGORY DISTRIBUTION BY REPOSITORY")
print("=" * 80)

pivot = class_df.pivot_table(
    index='repo_name', 
    columns='category', 
    aggfunc='size', 
    fill_value=0
)
pivot['TOTAL'] = pivot.sum(axis=1)
print(pivot.to_string())

## Phase 5: UMAP Visualization with All Repositories

Project all files and category centroids into 2D space together.

In [ ]:
# Fit UMAP on file embeddings only
# Centroid 2D positions will be computed from assigned files (avoids transform() crash)

print(f"File embeddings shape: {embeddings.shape}")

# Configure UMAP reducer
# n_jobs=1 prevents OpenMP threading crash on macOS
print("\nFitting UMAP on file embeddings only...")
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='cosine',
    random_state=42,
    n_jobs=1  # Prevent OMP threading crash
)

# Fit on 300 files only (same as embedding_natural_clusters.ipynb)
file_coords = reducer.fit_transform(embeddings)

print(f"\nUMAP complete:")
print(f"  File coords shape: {file_coords.shape}")

# NOTE: Centroid 2D positions will be computed in the next cell
# as the mean of assigned files' coordinates (avoids transform() crash)

In [ ]:
# Build DataFrame for file points
plot_df = pd.DataFrame({
    'global_file_id': all_file_ids,
    'repo_name': all_repo_labels,
    'umap_x': file_coords[:, 0],
    'umap_y': file_coords[:, 1],
    'category': class_df['category'].values,
    'confidence': class_df['confidence'].values,
    'second_category': class_df['second_category'].values,
    'second_confidence': class_df['second_confidence'].values,
    'is_ambiguous': class_df['is_ambiguous'].values,
    'is_low_confidence': class_df['is_low_confidence'].values,
})

# Merge with metadata
plot_df = plot_df.merge(
    metadata_df[['global_file_id', 'artifact_name', 'artifact_path', 'tool_name']], 
    on='global_file_id', 
    how='left'
)

# Compute centroid 2D positions from the mean of assigned files' coordinates
# This avoids the UMAP transform() call which crashes on macOS ARM
print("Computing centroid positions from assigned files...")
centroid_coords_list = []
for category in category_names:
    mask = (plot_df['category'] == category)
    if mask.any():
        centroid_x = plot_df.loc[mask, 'umap_x'].mean()
        centroid_y = plot_df.loc[mask, 'umap_y'].mean()
        n_files = mask.sum()
        print(f"  {category}: ({centroid_x:.2f}, {centroid_y:.2f}) from {n_files} files")
    else:
        # Fallback: place at center if no files assigned
        centroid_x = plot_df['umap_x'].mean()
        centroid_y = plot_df['umap_y'].mean()
        print(f"  {category}: ({centroid_x:.2f}, {centroid_y:.2f}) - NO FILES ASSIGNED, using center")
    centroid_coords_list.append([centroid_x, centroid_y])

centroid_coords = np.array(centroid_coords_list)

# Build DataFrame for centroid points
centroid_df = pd.DataFrame({
    'category': category_names,
    'umap_x': centroid_coords[:, 0],
    'umap_y': centroid_coords[:, 1],
})

print(f"\nPlot data ready:")
print(f"  Files: {len(plot_df)}")
print(f"  Centroids: {len(centroid_df)}")

## Phase 6: Interactive Visualizations

In [ ]:
# Color maps
category_colors = {
    'agents': '#1f77b4',       # blue
    'commands': '#ff7f0e',     # orange
    'flows': '#2ca02c',        # green
    'rules': '#d62728',        # red
    'skills': '#17becf',       # cyan
    'architecture': '#9467bd', # purple
    'code-style': '#8c564b',   # brown
    'configuration': '#e377c2', # pink
    'session-logs': '#bcbd22', # olive
}

# Generate distinct colors for repositories
import colorsys
repositories = sorted(metadata_df['repo_name'].unique())
n_repos = len(repositories)
repo_colors = {}
for i, repo in enumerate(repositories):
    hue = i / n_repos
    rgb = colorsys.hsv_to_rgb(hue, 0.7, 0.9)
    repo_colors[repo] = f'rgb({int(rgb[0]*255)}, {int(rgb[1]*255)}, {int(rgb[2]*255)})'

print(f"Category colors: {len(category_colors)}")
print(f"Repository colors: {len(repo_colors)}")

In [ ]:
# Visualization 1: Color by CATEGORY
fig1 = go.Figure()

# Add file points grouped by category
for category in sorted(plot_df['category'].unique()):
    cat_data = plot_df[plot_df['category'] == category]
    color = category_colors.get(category, '#7f7f7f')
    
    fig1.add_trace(go.Scatter(
        x=cat_data['umap_x'],
        y=cat_data['umap_y'],
        mode='markers',
        name=f"{category} ({len(cat_data)})",
        marker=dict(size=8, color=color, opacity=0.7, line=dict(width=1, color='white')),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>" +
            "Repo: %{customdata[1]}<br>" +
            "Path: %{customdata[2]}<br>" +
            "Category: %{customdata[3]} (conf: %{customdata[4]:.3f})<br>" +
            "2nd: %{customdata[5]}<br>" +
            "<extra></extra>"
        ),
        customdata=cat_data[['artifact_name', 'repo_name', 'artifact_path', 'category', 'confidence', 'second_category']].values,
        legendgroup=category,
    ))

# Add category centroids
for _, row in centroid_df.iterrows():
    category = row['category']
    color = category_colors.get(category, '#7f7f7f')
    
    fig1.add_trace(go.Scatter(
        x=[row['umap_x']],
        y=[row['umap_y']],
        mode='markers+text',
        marker=dict(size=20, color=color, symbol='star', line=dict(width=2, color='black')),
        text=[category.upper()],
        textposition='top center',
        textfont=dict(size=10, color='black'),
        hovertemplate=f"<b>{category.upper()} CENTROID</b><extra></extra>",
        legendgroup=category,
        showlegend=False,
    ))

fig1.update_layout(
    title=f'All Repositories - Colored by Category<br><sup>{len(plot_df)} files from {len(repositories)} repos, stars = category centroids</sup>',
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    width=1200,
    height=800,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02, title="Categories"),
    hovermode='closest',
)

fig1.show()

## Phase 6a: Embedding Quality Exploration

Before filtering, let's understand WHY confidence scores are low and margins are small. This analysis helps us decide:
- Are our category templates well-suited to the data?
- Which categories overlap most?
- Is the data genuinely ambiguous, or do we need better templates?

In [ ]:
# 6a.1: Confidence Score Distribution
# Understanding where the mass of data sits helps us set appropriate thresholds

fig_conf = make_subplots(rows=1, cols=2, subplot_titles=(
    'Confidence Score Distribution', 
    'Margin (Top1 - Top2) Distribution'
))

# Confidence histogram
fig_conf.add_trace(
    go.Histogram(x=plot_df['confidence'], nbinsx=50, name='Confidence',
                 marker_color='steelblue', opacity=0.7),
    row=1, col=1
)

# Add vertical lines for thresholds
fig_conf.add_vline(x=0.25, line_dash="dash", line_color="red", row=1, col=1,
                   annotation_text="default threshold (0.25)")
fig_conf.add_vline(x=0.30, line_dash="dash", line_color="orange", row=1, col=1,
                   annotation_text="strict threshold (0.30)")

# Margin histogram
margins = plot_df['confidence'] - plot_df['second_confidence']
fig_conf.add_trace(
    go.Histogram(x=margins, nbinsx=50, name='Margin',
                 marker_color='forestgreen', opacity=0.7),
    row=1, col=2
)

fig_conf.add_vline(x=0.05, line_dash="dash", line_color="red", row=1, col=2,
                   annotation_text="ambiguity threshold (0.05)")
fig_conf.add_vline(x=0.08, line_dash="dash", line_color="orange", row=1, col=2,
                   annotation_text="strict margin (0.08)")

fig_conf.update_layout(
    title=f'Classification Quality Distribution (n={len(plot_df):,} files)',
    height=400, width=1100,
    showlegend=False
)
fig_conf.update_xaxes(title_text="Confidence Score", row=1, col=1)
fig_conf.update_xaxes(title_text="Margin (Top1 - Top2)", row=1, col=2)
fig_conf.update_yaxes(title_text="Count", row=1, col=1)
fig_conf.update_yaxes(title_text="Count", row=1, col=2)

fig_conf.show()

# Print percentile analysis
print("=" * 70)
print("CONFIDENCE SCORE PERCENTILES")
print("=" * 70)
percentiles = [10, 25, 50, 75, 90, 95]
for p in percentiles:
    conf_val = np.percentile(plot_df['confidence'], p)
    margin_val = np.percentile(margins, p)
    print(f"  {p:3d}th percentile: confidence={conf_val:.3f}, margin={margin_val:.3f}")

# How much data at various thresholds
print(f"\nData retention at different confidence thresholds:")
for thresh in [0.20, 0.25, 0.30, 0.35, 0.40]:
    pct = (plot_df['confidence'] >= thresh).mean() * 100
    print(f"  conf >= {thresh}: {pct:.1f}% retained")

In [ ]:
# 6a.2: Category Confusion Analysis
# When files are ambiguous, which categories compete with each other?

# Build confusion matrix: for each assigned category, what's the most common second choice?
confusion_counts = pd.crosstab(
    plot_df['category'], 
    plot_df['second_category'],
    margins=True
)

print("=" * 70)
print("CATEGORY CONFUSION MATRIX")
print("When a file is assigned to [row], its second choice is [column]")
print("=" * 70)
print(confusion_counts.to_string())

# Heatmap visualization (excluding margins row/col for clarity)
confusion_matrix = pd.crosstab(plot_df['category'], plot_df['second_category'])

fig_confusion = go.Figure(data=go.Heatmap(
    z=confusion_matrix.values,
    x=confusion_matrix.columns,
    y=confusion_matrix.index,
    colorscale='Blues',
    text=confusion_matrix.values,
    texttemplate='%{text}',
    textfont={"size": 10},
    hovertemplate='Assigned: %{y}<br>Second choice: %{x}<br>Count: %{z}<extra></extra>',
))

fig_confusion.update_layout(
    title='Category Confusion Matrix<br><sup>Row = assigned category, Column = second-best category</sup>',
    xaxis_title='Second Choice Category',
    yaxis_title='Assigned Category',
    width=800,
    height=600,
)

fig_confusion.show()

# Most confused pairs
print("\nMost Confused Category Pairs (bidirectional):")
print("-" * 50)
pair_confusion = {}
for cat1 in category_names:
    for cat2 in category_names:
        if cat1 < cat2:  # Avoid duplicates
            count1 = confusion_matrix.loc[cat1, cat2] if cat2 in confusion_matrix.columns and cat1 in confusion_matrix.index else 0
            count2 = confusion_matrix.loc[cat2, cat1] if cat1 in confusion_matrix.columns and cat2 in confusion_matrix.index else 0
            pair_confusion[(cat1, cat2)] = count1 + count2

for (cat1, cat2), count in sorted(pair_confusion.items(), key=lambda x: -x[1])[:10]:
    total_cat1 = (plot_df['category'] == cat1).sum()
    total_cat2 = (plot_df['category'] == cat2).sum()
    print(f"  {cat1} <-> {cat2}: {count:,} files ({count/(total_cat1+total_cat2)*100:.1f}% of both categories)")

In [ ]:
# 6a.3: Per-Category Quality Analysis
# Which categories have the strongest/weakest classifications?

# Box plot of confidence by category
fig_box = go.Figure()

for category in category_names:
    cat_conf = plot_df[plot_df['category'] == category]['confidence']
    color = category_colors.get(category, '#7f7f7f')
    
    fig_box.add_trace(go.Box(
        y=cat_conf,
        name=f"{category}<br>(n={len(cat_conf):,})",
        marker_color=color,
        boxpoints='outliers',
    ))

fig_box.add_hline(y=0.25, line_dash="dash", line_color="red",
                  annotation_text="default threshold")
fig_box.add_hline(y=0.30, line_dash="dash", line_color="orange",
                  annotation_text="strict threshold")

fig_box.update_layout(
    title='Confidence Score Distribution by Category',
    yaxis_title='Confidence Score',
    height=500,
    width=1000,
    showlegend=False,
)

fig_box.show()

# Detailed stats per category
print("=" * 90)
print("PER-CATEGORY CLASSIFICATION QUALITY")
print("=" * 90)
print(f"{'Category':<15} {'Count':>8} {'Mean Conf':>10} {'Median':>8} {'Std':>8} {'% >= 0.25':>10} {'% >= 0.30':>10}")
print("-" * 90)

category_stats = []
for category in category_names:
    cat_data = plot_df[plot_df['category'] == category]
    cat_conf = cat_data['confidence']
    
    stats = {
        'category': category,
        'count': len(cat_conf),
        'mean': cat_conf.mean(),
        'median': cat_conf.median(),
        'std': cat_conf.std(),
        'pct_above_25': (cat_conf >= 0.25).mean() * 100,
        'pct_above_30': (cat_conf >= 0.30).mean() * 100,
    }
    category_stats.append(stats)
    
    print(f"{category:<15} {stats['count']:>8,} {stats['mean']:>10.3f} {stats['median']:>8.3f} "
          f"{stats['std']:>8.3f} {stats['pct_above_25']:>9.1f}% {stats['pct_above_30']:>9.1f}%")

# Identify problematic categories
print("\n" + "=" * 90)
print("CATEGORY HEALTH ASSESSMENT")
print("=" * 90)
for stats in sorted(category_stats, key=lambda x: x['mean']):
    if stats['pct_above_30'] < 30:
        print(f"⚠️  {stats['category']}: Only {stats['pct_above_30']:.1f}% above 0.30 threshold - may need template refinement")
    elif stats['pct_above_30'] < 50:
        print(f"🔶 {stats['category']}: {stats['pct_above_30']:.1f}% above 0.30 - moderate quality")
    else:
        print(f"✅ {stats['category']}: {stats['pct_above_30']:.1f}% above 0.30 - good separation")

In [ ]:
# 6a.4: Sample Review of Borderline Files
# What do files with confidence 0.20-0.30 look like? Are they genuinely ambiguous?

print("=" * 90)
print("SAMPLE BORDERLINE FILES (confidence 0.20 - 0.30)")
print("These files would be filtered out with strict threshold but kept with default")
print("=" * 90)

borderline = plot_df[(plot_df['confidence'] >= 0.20) & (plot_df['confidence'] < 0.30)].copy()
borderline['margin'] = borderline['confidence'] - borderline['second_confidence']

print(f"\nTotal borderline files: {len(borderline):,}")
print(f"\nSample of 15 borderline files:\n")

# Sample across categories
sample_size = min(15, len(borderline))
if len(borderline) > 0:
    sample = borderline.sample(n=sample_size, random_state=42)
    
    for _, row in sample.iterrows():
        print(f"  📄 {row['artifact_name']}")
        print(f"     Path: {row['artifact_path']}")
        print(f"     Repo: {row['repo_name']}")
        print(f"     Assigned: {row['category']} (conf: {row['confidence']:.3f})")
        print(f"     2nd choice: {row['second_category']} (conf: {row['second_confidence']:.3f}, margin: {row['margin']:.3f})")
        print()

# Also show very low confidence files
print("\n" + "=" * 90)
print("SAMPLE VERY LOW CONFIDENCE FILES (confidence < 0.20)")
print("=" * 90)

very_low = plot_df[plot_df['confidence'] < 0.20].copy()
very_low['margin'] = very_low['confidence'] - very_low['second_confidence']

print(f"\nTotal very low confidence files: {len(very_low):,}")

if len(very_low) > 0:
    sample_low = very_low.sample(n=min(10, len(very_low)), random_state=42)
    print(f"\nSample of {len(sample_low)} very low confidence files:\n")
    
    for _, row in sample_low.iterrows():
        print(f"  📄 {row['artifact_name']}")
        print(f"     Assigned: {row['category']} (conf: {row['confidence']:.3f})")
        print(f"     2nd: {row['second_category']} (margin: {row['margin']:.3f})")
        print()

In [ ]:
# 6a.5: Embedding Space Analysis
# Are files inherently similar to each other (tight clustering)?
# This would explain low margins regardless of category definitions.

from sklearn.metrics.pairwise import pairwise_distances

print("=" * 70)
print("EMBEDDING SPACE DENSITY ANALYSIS")
print("=" * 70)

# Sample embeddings for efficiency (full pairwise is expensive for 19k files)
sample_size = min(2000, len(embeddings))
sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)
sample_embeddings = embeddings[sample_indices]

# Compute pairwise cosine distances
print(f"\nComputing pairwise distances on {sample_size} sampled files...")
distances = pairwise_distances(sample_embeddings, metric='cosine')

# Get upper triangle (exclude diagonal)
upper_tri = distances[np.triu_indices(len(distances), k=1)]

print(f"\nPairwise Cosine Distance Statistics:")
print(f"  Mean distance: {upper_tri.mean():.3f}")
print(f"  Std distance:  {upper_tri.std():.3f}")
print(f"  Min distance:  {upper_tri.min():.3f}")
print(f"  Max distance:  {upper_tri.max():.3f}")
print(f"  Median:        {np.median(upper_tri):.3f}")

# Convert to similarity for interpretation
similarities = 1 - upper_tri
print(f"\nPairwise Cosine Similarity Statistics:")
print(f"  Mean similarity: {similarities.mean():.3f}")
print(f"  Std similarity:  {similarities.std():.3f}")

# Distance distribution histogram
fig_dist = go.Figure()
fig_dist.add_trace(go.Histogram(
    x=upper_tri,
    nbinsx=50,
    marker_color='purple',
    opacity=0.7,
))

fig_dist.update_layout(
    title=f'Pairwise Cosine Distance Distribution<br><sup>Sampled {sample_size} files - lower values = more similar</sup>',
    xaxis_title='Cosine Distance',
    yaxis_title='Count',
    height=400,
    width=800,
)

fig_dist.show()

# Interpretation
if similarities.mean() > 0.7:
    print("\n⚠️  HIGH EMBEDDING SIMILARITY: Files are very similar in embedding space.")
    print("    This explains low margins - all categories have similar overlap.")
    print("    Consider: More specific category templates, or accept lower thresholds.")
elif similarities.mean() > 0.5:
    print("\n🔶 MODERATE EMBEDDING SIMILARITY: Some natural clustering exists.")
    print("    Categories with high confusion may need template refinement.")
else:
    print("\n✅ GOOD EMBEDDING SPREAD: Files are reasonably spread in embedding space.")
    print("    Low confidence likely due to category template mismatch.")

In [ ]:
# 6a.6: Analysis Summary & Recommendations
print("=" * 70)
print("EXPLORATION SUMMARY & RECOMMENDATIONS")
print("=" * 70)

# Calculate key metrics
median_conf = plot_df['confidence'].median()
pct_above_25 = (plot_df['confidence'] >= 0.25).mean() * 100
pct_above_30 = (plot_df['confidence'] >= 0.30).mean() * 100
margins = plot_df['confidence'] - plot_df['second_confidence']
median_margin = margins.median()

print(f"\n📊 KEY METRICS:")
print(f"   Median confidence: {median_conf:.3f}")
print(f"   Median margin:     {median_margin:.3f}")
print(f"   Files >= 0.25:     {pct_above_25:.1f}%")
print(f"   Files >= 0.30:     {pct_above_30:.1f}%")

print(f"\n💡 RECOMMENDATIONS:")

# Recommendation based on confidence distribution
if median_conf < 0.25:
    print(f"\n   1. LOWER THRESHOLD: Median confidence ({median_conf:.3f}) is below 0.25.")
    print(f"      → Consider using 0.20 as threshold to retain more data.")
    print(f"      → Or refine category templates to better match your data.")
elif median_conf < 0.30:
    print(f"\n   1. MODERATE THRESHOLD: Median confidence ({median_conf:.3f}) is between 0.25-0.30.")
    print(f"      → Use 0.25 threshold for balanced retention.")
    print(f"      → Consider template refinement for low-performing categories.")
else:
    print(f"\n   1. GOOD CONFIDENCE: Median confidence ({median_conf:.3f}) is healthy.")
    print(f"      → 0.30 threshold is reasonable.")

# Recommendation based on margin
if median_margin < 0.05:
    print(f"\n   2. LOW MARGINS: Median margin ({median_margin:.3f}) indicates high category overlap.")
    print(f"      → Consider merging highly confused category pairs.")
    print(f"      → Or use margin threshold of 0.03-0.04 instead of 0.08.")
elif median_margin < 0.08:
    print(f"\n   2. MODERATE MARGINS: Margin ({median_margin:.3f}) is below strict threshold.")
    print(f"      → Use 0.05 margin threshold instead of 0.08.")
else:
    print(f"\n   2. GOOD MARGINS: Median margin ({median_margin:.3f}) shows decent separation.")

# Suggest adjusted thresholds
suggested_conf = max(0.20, np.percentile(plot_df['confidence'], 25))  # 25th percentile
suggested_margin = max(0.03, np.percentile(margins, 25))  # 25th percentile

print(f"\n   📈 SUGGESTED THRESHOLDS (to retain ~75% of data):")
print(f"      - Confidence: {suggested_conf:.2f}")
print(f"      - Margin:     {suggested_margin:.2f}")

# Estimate retention with suggested thresholds
mask_suggested = (plot_df['confidence'] >= suggested_conf) & (margins >= suggested_margin)
suggested_retention = mask_suggested.mean() * 100
print(f"      - Estimated retention: {suggested_retention:.1f}%")

print("\n" + "=" * 70)

### 6a.7: Distance from Centroid Analysis

How far is each file from its assigned category centroid in the **original embedding space** (768 dimensions)?

Files far from their centroid may be misclassified or genuinely ambiguous artifacts.

In [ ]:
# 6a.7: Distance from Centroid Analysis
# Compute cosine distance from each file to its ASSIGNED category centroid
# This is different from confidence score - it's the actual geometric distance

print("=" * 70)
print("DISTANCE FROM CENTROID ANALYSIS")
print("=" * 70)

# Build category centroid matrix (in original embedding space)
category_centroid_matrix = np.vstack([category_embeddings[cat] for cat in category_names])
category_to_idx = {cat: i for i, cat in enumerate(category_names)}

# For each file, compute distance to its assigned centroid
centroid_distances = []
for i, row in plot_df.iterrows():
    file_embedding = embeddings[i].reshape(1, -1)
    assigned_cat = row['category']
    cat_idx = category_to_idx[assigned_cat]
    centroid_embedding = category_centroid_matrix[cat_idx].reshape(1, -1)
    
    # Cosine distance = 1 - cosine_similarity
    cos_sim = cosine_similarity(file_embedding, centroid_embedding)[0, 0]
    cos_dist = 1 - cos_sim
    centroid_distances.append(cos_dist)

plot_df['centroid_distance'] = centroid_distances

print(f"\nCentroid Distance Statistics:")
print(f"  Min:    {plot_df['centroid_distance'].min():.3f}")
print(f"  Max:    {plot_df['centroid_distance'].max():.3f}")
print(f"  Mean:   {plot_df['centroid_distance'].mean():.3f}")
print(f"  Median: {plot_df['centroid_distance'].median():.3f}")
print(f"  Std:    {plot_df['centroid_distance'].std():.3f}")

# Histogram of centroid distances
fig_cd = go.Figure()
fig_cd.add_trace(go.Histogram(
    x=plot_df['centroid_distance'],
    nbinsx=50,
    marker_color='coral',
    opacity=0.7,
))

# Add threshold lines
for thresh, color, name in [(0.7, 'red', 'strict'), (0.75, 'orange', 'moderate'), (0.8, 'green', 'loose')]:
    fig_cd.add_vline(x=thresh, line_dash="dash", line_color=color,
                     annotation_text=f"{name} ({thresh})")

fig_cd.update_layout(
    title='Distance from Assigned Centroid<br><sup>Lower = closer to category center, higher = outlier</sup>',
    xaxis_title='Cosine Distance to Centroid',
    yaxis_title='Count',
    height=400,
    width=900,
)
fig_cd.show()

# Retention at different thresholds
print(f"\nData retention at different centroid distance thresholds:")
for thresh in [0.65, 0.70, 0.75, 0.80, 0.85]:
    pct = (plot_df['centroid_distance'] <= thresh).mean() * 100
    count = (plot_df['centroid_distance'] <= thresh).sum()
    print(f"  distance <= {thresh}: {pct:.1f}% retained ({count:,} files)")

In [ ]:
# 6a.7b: Centroid Distance by Category
# Are some categories tighter than others?

fig_cd_cat = go.Figure()

for category in category_names:
    cat_dist = plot_df[plot_df['category'] == category]['centroid_distance']
    color = category_colors.get(category, '#7f7f7f')
    
    fig_cd_cat.add_trace(go.Box(
        y=cat_dist,
        name=f"{category}<br>(n={len(cat_dist):,})",
        marker_color=color,
        boxpoints='outliers',
    ))

fig_cd_cat.add_hline(y=0.75, line_dash="dash", line_color="red",
                     annotation_text="suggested threshold")

fig_cd_cat.update_layout(
    title='Centroid Distance by Category<br><sup>Lower = tighter cluster, higher = more dispersed</sup>',
    yaxis_title='Cosine Distance to Centroid',
    height=500,
    width=1000,
    showlegend=False,
)
fig_cd_cat.show()

# Per-category stats
print("\nPer-Category Centroid Distance Stats:")
print(f"{'Category':<15} {'Mean':>8} {'Median':>8} {'Std':>8} {'% <= 0.75':>10}")
print("-" * 55)
for category in category_names:
    cat_dist = plot_df[plot_df['category'] == category]['centroid_distance']
    pct_close = (cat_dist <= 0.75).mean() * 100
    print(f"{category:<15} {cat_dist.mean():>8.3f} {cat_dist.median():>8.3f} {cat_dist.std():>8.3f} {pct_close:>9.1f}%")

### 6a.8: HDBSCAN Density Analysis

Use density-based clustering to identify:
- **Core samples**: Points in dense regions (likely real clusters)
- **Noise points**: Isolated points that don't belong to any dense region

HDBSCAN doesn't force every point into a cluster - noise points get label `-1`.

In [ ]:
# 6a.8: HDBSCAN Density Analysis
import hdbscan

print("=" * 70)
print("HDBSCAN DENSITY ANALYSIS")
print("=" * 70)

# Run HDBSCAN on UMAP 2D coordinates (much faster than 768-dim embeddings)
# This finds density clusters in the visualization space
print("\nRunning HDBSCAN on UMAP 2D coordinates (fast)...")

# Use the already-computed UMAP coordinates
umap_coords = plot_df[['umap_x', 'umap_y']].values

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
)
clusterer.fit(umap_coords)

# Get labels and probabilities
plot_df['hdbscan_label'] = clusterer.labels_
plot_df['hdbscan_prob'] = clusterer.probabilities_
plot_df['hdbscan_outlier_score'] = clusterer.outlier_scores_

# Summary
n_clusters = len(set(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0)
n_noise = (clusterer.labels_ == -1).sum()
noise_pct = n_noise / len(clusterer.labels_) * 100

print(f"\nHDBSCAN Results:")
print(f"  Clusters found: {n_clusters}")
print(f"  Noise points:   {n_noise:,} ({noise_pct:.1f}%)")
print(f"  Core points:    {len(clusterer.labels_) - n_noise:,} ({100-noise_pct:.1f}%)")

# Cluster size distribution
print(f"\nCluster sizes:")
for label in sorted(set(clusterer.labels_)):
    count = (clusterer.labels_ == label).sum()
    pct = count / len(clusterer.labels_) * 100
    label_name = "NOISE" if label == -1 else f"Cluster {label}"
    print(f"  {label_name}: {count:,} ({pct:.1f}%)")

In [ ]:
# 6a.8b: HDBSCAN Outlier Score Distribution
# Higher outlier score = more likely to be noise

fig_hdb = make_subplots(rows=1, cols=2, subplot_titles=(
    'HDBSCAN Outlier Score Distribution',
    'HDBSCAN Membership Probability'
))

# Outlier score histogram
fig_hdb.add_trace(
    go.Histogram(x=plot_df['hdbscan_outlier_score'], nbinsx=50,
                 marker_color='crimson', opacity=0.7),
    row=1, col=1
)

# Membership probability histogram
fig_hdb.add_trace(
    go.Histogram(x=plot_df['hdbscan_prob'], nbinsx=50,
                 marker_color='teal', opacity=0.7),
    row=1, col=2
)

# Threshold lines
fig_hdb.add_vline(x=0.8, line_dash="dash", line_color="red", row=1, col=1,
                  annotation_text="high outlier (0.8)")
fig_hdb.add_vline(x=0.5, line_dash="dash", line_color="orange", row=1, col=1,
                  annotation_text="moderate (0.5)")

fig_hdb.add_vline(x=0.1, line_dash="dash", line_color="red", row=1, col=2,
                  annotation_text="low prob (0.1)")
fig_hdb.add_vline(x=0.5, line_dash="dash", line_color="orange", row=1, col=2,
                  annotation_text="moderate (0.5)")

fig_hdb.update_layout(
    title=f'HDBSCAN Density Metrics (n={len(plot_df):,} files)',
    height=400, width=1100,
    showlegend=False
)
fig_hdb.update_xaxes(title_text="Outlier Score (higher=more outlier)", row=1, col=1)
fig_hdb.update_xaxes(title_text="Membership Probability (higher=more core)", row=1, col=2)
fig_hdb.update_yaxes(title_text="Count", row=1, col=1)
fig_hdb.update_yaxes(title_text="Count", row=1, col=2)

fig_hdb.show()

# Retention at different outlier score thresholds
print(f"\nData retention at different outlier score thresholds:")
for thresh in [0.3, 0.5, 0.7, 0.8, 0.9]:
    pct = (plot_df['hdbscan_outlier_score'] <= thresh).mean() * 100
    count = (plot_df['hdbscan_outlier_score'] <= thresh).sum()
    print(f"  outlier_score <= {thresh}: {pct:.1f}% retained ({count:,} files)")

In [ ]:
# 6a.8c: How do HDBSCAN clusters align with our predefined categories?

print("\nHDBSCAN Cluster vs Predefined Category Cross-tabulation:")
crosstab = pd.crosstab(plot_df['hdbscan_label'], plot_df['category'], margins=True)
print(crosstab.to_string())

# What percentage of each category is marked as noise by HDBSCAN?
print(f"\nPercentage of each category marked as NOISE by HDBSCAN:")
print(f"{'Category':<15} {'Total':>8} {'Noise':>8} {'% Noise':>10}")
print("-" * 45)
for category in category_names:
    cat_mask = plot_df['category'] == category
    total = cat_mask.sum()
    noise = ((plot_df['hdbscan_label'] == -1) & cat_mask).sum()
    pct = noise / total * 100 if total > 0 else 0
    print(f"{category:<15} {total:>8,} {noise:>8,} {pct:>9.1f}%")

### 6a.9: Local Density Analysis

For each file, compute the **mean distance to its k-nearest neighbors**.

- **Low mean distance** = point is in a dense region (has close neighbors)
- **High mean distance** = point is isolated (potential outlier)

In [ ]:
# 6a.9: Local Density Analysis
from sklearn.neighbors import NearestNeighbors

print("=" * 70)
print("LOCAL DENSITY ANALYSIS")
print("=" * 70)

# Compute k-nearest neighbors for each point
K = 10  # number of neighbors to consider
print(f"\nComputing {K}-nearest neighbors for each file...")

nn = NearestNeighbors(n_neighbors=K+1, metric='cosine')  # +1 because point is its own neighbor
nn.fit(embeddings)
distances, indices = nn.kneighbors(embeddings)

# Mean distance to k-nearest neighbors (excluding self)
mean_knn_distance = distances[:, 1:].mean(axis=1)  # exclude first column (self)
plot_df['local_density_dist'] = mean_knn_distance

# Convert to density score (inverse of distance)
# Higher density = lower distance to neighbors
plot_df['local_density_score'] = 1 / (1 + mean_knn_distance)

print(f"\nLocal Density Distance Statistics (mean dist to {K} nearest neighbors):")
print(f"  Min:    {plot_df['local_density_dist'].min():.4f}")
print(f"  Max:    {plot_df['local_density_dist'].max():.4f}")
print(f"  Mean:   {plot_df['local_density_dist'].mean():.4f}")
print(f"  Median: {plot_df['local_density_dist'].median():.4f}")
print(f"  Std:    {plot_df['local_density_dist'].std():.4f}")

In [ ]:
# 6a.9b: Local Density Distribution

fig_ld = make_subplots(rows=1, cols=2, subplot_titles=(
    f'Mean Distance to {K} Nearest Neighbors',
    'Local Density Score (1/(1+dist))'
))

# Distance histogram
fig_ld.add_trace(
    go.Histogram(x=plot_df['local_density_dist'], nbinsx=50,
                 marker_color='darkorange', opacity=0.7),
    row=1, col=1
)

# Density score histogram
fig_ld.add_trace(
    go.Histogram(x=plot_df['local_density_score'], nbinsx=50,
                 marker_color='seagreen', opacity=0.7),
    row=1, col=2
)

# Threshold lines for distance (higher = more isolated)
median_dist = plot_df['local_density_dist'].median()
p75_dist = plot_df['local_density_dist'].quantile(0.75)
p90_dist = plot_df['local_density_dist'].quantile(0.90)

fig_ld.add_vline(x=p75_dist, line_dash="dash", line_color="orange", row=1, col=1,
                 annotation_text=f"75th pct ({p75_dist:.3f})")
fig_ld.add_vline(x=p90_dist, line_dash="dash", line_color="red", row=1, col=1,
                 annotation_text=f"90th pct ({p90_dist:.3f})")

fig_ld.update_layout(
    title=f'Local Density Analysis (K={K} neighbors)',
    height=400, width=1100,
    showlegend=False
)
fig_ld.update_xaxes(title_text="Mean Distance to Neighbors (higher=isolated)", row=1, col=1)
fig_ld.update_xaxes(title_text="Density Score (higher=dense region)", row=1, col=2)
fig_ld.update_yaxes(title_text="Count", row=1, col=1)
fig_ld.update_yaxes(title_text="Count", row=1, col=2)

fig_ld.show()

# Retention at different thresholds
print(f"\nData retention at different local density distance thresholds:")
for pct in [50, 60, 70, 75, 80, 90]:
    thresh = plot_df['local_density_dist'].quantile(pct/100)
    retained = (plot_df['local_density_dist'] <= thresh).mean() * 100
    count = (plot_df['local_density_dist'] <= thresh).sum()
    print(f"  <= {pct}th percentile (dist <= {thresh:.4f}): {retained:.1f}% retained ({count:,} files)")

In [ ]:
# 6a.9c: Local Density by Category

fig_ld_cat = go.Figure()

for category in category_names:
    cat_dens = plot_df[plot_df['category'] == category]['local_density_dist']
    color = category_colors.get(category, '#7f7f7f')
    
    fig_ld_cat.add_trace(go.Box(
        y=cat_dens,
        name=f"{category}<br>(n={len(cat_dens):,})",
        marker_color=color,
        boxpoints='outliers',
    ))

p75_dist = plot_df['local_density_dist'].quantile(0.75)
fig_ld_cat.add_hline(y=p75_dist, line_dash="dash", line_color="red",
                     annotation_text=f"75th percentile ({p75_dist:.3f})")

fig_ld_cat.update_layout(
    title='Local Density Distance by Category<br><sup>Higher = more isolated points in that category</sup>',
    yaxis_title='Mean Distance to K Nearest Neighbors',
    height=500,
    width=1000,
    showlegend=False,
)
fig_ld_cat.show()

# Per-category stats
print(f"\nPer-Category Local Density Stats (mean distance to {K} neighbors):")
print(f"{'Category':<15} {'Mean':>10} {'Median':>10} {'% <= 75th':>12}")
print("-" * 50)
for category in category_names:
    cat_dens = plot_df[plot_df['category'] == category]['local_density_dist']
    pct_dense = (cat_dens <= p75_dist).mean() * 100
    print(f"{category:<15} {cat_dens.mean():>10.4f} {cat_dens.median():>10.4f} {pct_dense:>11.1f}%")

### 6a.10: Combined Filter Exploration

Now that we have multiple filtering metrics, let's explore how they correlate and what combinations work best.

In [ ]:
# 6a.10: Correlation between filtering metrics

filter_cols = ['confidence', 'centroid_distance', 'hdbscan_outlier_score', 'local_density_dist']
corr_matrix = plot_df[filter_cols].corr()

print("=" * 70)
print("FILTER METRIC CORRELATIONS")
print("=" * 70)
print(corr_matrix.round(3).to_string())

# Heatmap
fig_corr = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=['Confidence', 'Centroid Dist', 'HDBSCAN Outlier', 'Local Density Dist'],
    y=['Confidence', 'Centroid Dist', 'HDBSCAN Outlier', 'Local Density Dist'],
    colorscale='RdBu_r',
    zmid=0,
    text=corr_matrix.round(2).values,
    texttemplate='%{text}',
    textfont={"size": 12},
))

fig_corr.update_layout(
    title='Correlation Between Filter Metrics',
    width=600,
    height=500,
)
fig_corr.show()

In [ ]:
# 6a.10b: Scatter plots - Confidence vs other metrics

fig_scatter = make_subplots(rows=1, cols=3, subplot_titles=(
    'Confidence vs Centroid Distance',
    'Confidence vs HDBSCAN Outlier Score',
    'Confidence vs Local Density Distance'
))

# Sample for performance
sample_size = min(3000, len(plot_df))
sample_df = plot_df.sample(n=sample_size, random_state=42)

fig_scatter.add_trace(
    go.Scatter(x=sample_df['confidence'], y=sample_df['centroid_distance'],
               mode='markers', marker=dict(size=4, opacity=0.5, color='coral'),
               showlegend=False),
    row=1, col=1
)

fig_scatter.add_trace(
    go.Scatter(x=sample_df['confidence'], y=sample_df['hdbscan_outlier_score'],
               mode='markers', marker=dict(size=4, opacity=0.5, color='crimson'),
               showlegend=False),
    row=1, col=2
)

fig_scatter.add_trace(
    go.Scatter(x=sample_df['confidence'], y=sample_df['local_density_dist'],
               mode='markers', marker=dict(size=4, opacity=0.5, color='darkorange'),
               showlegend=False),
    row=1, col=3
)

fig_scatter.update_xaxes(title_text="Confidence", row=1, col=1)
fig_scatter.update_xaxes(title_text="Confidence", row=1, col=2)
fig_scatter.update_xaxes(title_text="Confidence", row=1, col=3)
fig_scatter.update_yaxes(title_text="Centroid Distance", row=1, col=1)
fig_scatter.update_yaxes(title_text="HDBSCAN Outlier Score", row=1, col=2)
fig_scatter.update_yaxes(title_text="Local Density Distance", row=1, col=3)

fig_scatter.update_layout(
    title=f'Filter Metric Relationships (sampled {sample_size:,} files)',
    height=400, width=1200,
)
fig_scatter.show()

In [ ]:
# 6a.10c: Test combined filter thresholds

print("=" * 70)
print("COMBINED FILTER THRESHOLD EXPLORATION")
print("=" * 70)

# Define threshold combinations to test
threshold_combos = [
    {'name': 'Current (confidence + margin)', 'confidence': 0.22, 'margin': 0.03, 'centroid_dist': None, 'outlier': None, 'local_dist': None},
    {'name': 'Add centroid distance', 'confidence': 0.22, 'margin': 0.03, 'centroid_dist': 0.75, 'outlier': None, 'local_dist': None},
    {'name': 'Add HDBSCAN (no noise)', 'confidence': 0.22, 'margin': 0.03, 'centroid_dist': None, 'outlier': 0.8, 'local_dist': None},
    {'name': 'Add local density', 'confidence': 0.22, 'margin': 0.03, 'centroid_dist': None, 'outlier': None, 'local_dist': plot_df['local_density_dist'].quantile(0.75)},
    {'name': 'Centroid + HDBSCAN', 'confidence': 0.22, 'margin': 0.03, 'centroid_dist': 0.75, 'outlier': 0.8, 'local_dist': None},
    {'name': 'All filters (moderate)', 'confidence': 0.22, 'margin': 0.03, 'centroid_dist': 0.78, 'outlier': 0.7, 'local_dist': plot_df['local_density_dist'].quantile(0.80)},
    {'name': 'All filters (strict)', 'confidence': 0.25, 'margin': 0.05, 'centroid_dist': 0.75, 'outlier': 0.6, 'local_dist': plot_df['local_density_dist'].quantile(0.70)},
]

print(f"\n{'Filter Combo':<30} {'Retained':>10} {'Pct':>8}")
print("-" * 52)

for combo in threshold_combos:
    mask = pd.Series([True] * len(plot_df))
    
    # Apply each filter if threshold is set
    mask &= plot_df['confidence'] >= combo['confidence']
    mask &= (plot_df['confidence'] - plot_df['second_confidence']) >= combo['margin']
    
    if combo['centroid_dist'] is not None:
        mask &= plot_df['centroid_distance'] <= combo['centroid_dist']
    if combo['outlier'] is not None:
        mask &= plot_df['hdbscan_outlier_score'] <= combo['outlier']
    if combo['local_dist'] is not None:
        mask &= plot_df['local_density_dist'] <= combo['local_dist']
    
    retained = mask.sum()
    pct = retained / len(plot_df) * 100
    print(f"{combo['name']:<30} {retained:>10,} {pct:>7.1f}%")

## Phase 6b: Noise Filtering

Apply confidence + clarity filtering to remove noisy/ambiguous classifications.

In [ ]:
# Phase 6b: Combined Noise Filtering
# Using two complementary filters: Confidence+Margin AND HDBSCAN Outlier Score

print("=" * 70)
print("PHASE 6b: COMBINED NOISE FILTERING")
print("=" * 70)

# =============================================================================
# FILTER PARAMETERS
# =============================================================================
FILTER_CONFIDENCE_MIN = 0.22   # Minimum confidence score
FILTER_MARGIN_MIN = 0.03       # Minimum margin between top-1 and top-2 categories
FILTER_HDBSCAN_OUTLIER_MAX = 0.7  # Maximum HDBSCAN outlier score (lower = more core-like)

print(f"\nFilter Thresholds:")
print(f"  - Confidence >= {FILTER_CONFIDENCE_MIN}")
print(f"  - Margin >= {FILTER_MARGIN_MIN}")
print(f"  - HDBSCAN Outlier Score <= {FILTER_HDBSCAN_OUTLIER_MAX}")

# =============================================================================
# BEFORE FILTERING - BASELINE STATISTICS
# =============================================================================
print(f"\n{'='*70}")
print("BEFORE FILTERING (Baseline)")
print(f"{'='*70}")

n_before = len(plot_df)
n_repos_before = plot_df['repo_name'].nunique()
mean_conf_before = plot_df['confidence'].mean()
median_conf_before = plot_df['confidence'].median()
margins_before = plot_df['confidence'] - plot_df['second_confidence']
mean_margin_before = margins_before.mean()
pct_ambiguous_before = (margins_before < 0.05).mean() * 100
pct_noise_before = (plot_df['hdbscan_label'] == -1).mean() * 100

print(f"  Total files:           {n_before:,}")
print(f"  Total repositories:    {n_repos_before:,}")
print(f"  Mean confidence:       {mean_conf_before:.3f}")
print(f"  Median confidence:     {median_conf_before:.3f}")
print(f"  Mean margin:           {mean_margin_before:.3f}")
print(f"  % ambiguous (margin<0.05): {pct_ambiguous_before:.1f}%")
print(f"  % HDBSCAN noise:       {pct_noise_before:.1f}%")

# Category distribution before
print(f"\nCategory distribution (before):")
cat_counts_before = plot_df['category'].value_counts()
for cat, count in cat_counts_before.items():
    pct = count / n_before * 100
    print(f"    {cat}: {count:,} ({pct:.1f}%)")

# =============================================================================
# APPLY COMBINED FILTERS
# =============================================================================
print(f"\n{'='*70}")
print("APPLYING FILTERS")
print(f"{'='*70}")

# Filter 1: Confidence + Margin
mask_confidence = plot_df['confidence'] >= FILTER_CONFIDENCE_MIN
mask_margin = (plot_df['confidence'] - plot_df['second_confidence']) >= FILTER_MARGIN_MIN
mask_conf_margin = mask_confidence & mask_margin

# Filter 2: HDBSCAN Outlier Score
mask_hdbscan = plot_df['hdbscan_outlier_score'] <= FILTER_HDBSCAN_OUTLIER_MAX

# Combined filter
mask_combined = mask_conf_margin & mask_hdbscan

# Show filter breakdown
print(f"\nFilter breakdown:")
print(f"  Confidence >= {FILTER_CONFIDENCE_MIN}:     {mask_confidence.sum():,} pass ({mask_confidence.mean()*100:.1f}%)")
print(f"  Margin >= {FILTER_MARGIN_MIN}:            {mask_margin.sum():,} pass ({mask_margin.mean()*100:.1f}%)")
print(f"  Confidence + Margin:       {mask_conf_margin.sum():,} pass ({mask_conf_margin.mean()*100:.1f}%)")
print(f"  HDBSCAN outlier <= {FILTER_HDBSCAN_OUTLIER_MAX}:  {mask_hdbscan.sum():,} pass ({mask_hdbscan.mean()*100:.1f}%)")
print(f"  ----------------------------------------")
print(f"  COMBINED (all filters):    {mask_combined.sum():,} pass ({mask_combined.mean()*100:.1f}%)")

# Create filtered dataframe
plot_df_filtered = plot_df[mask_combined].copy()

# =============================================================================
# AFTER FILTERING - IMPACT STATISTICS
# =============================================================================
print(f"\n{'='*70}")
print("AFTER FILTERING (Impact)")
print(f"{'='*70}")

n_after = len(plot_df_filtered)
n_repos_after = plot_df_filtered['repo_name'].nunique()
mean_conf_after = plot_df_filtered['confidence'].mean()
median_conf_after = plot_df_filtered['confidence'].median()
margins_after = plot_df_filtered['confidence'] - plot_df_filtered['second_confidence']
mean_margin_after = margins_after.mean()
pct_ambiguous_after = (margins_after < 0.05).mean() * 100
pct_noise_after = (plot_df_filtered['hdbscan_label'] == -1).mean() * 100

print(f"  Total files:           {n_after:,}")
print(f"  Total repositories:    {n_repos_after:,}")
print(f"  Mean confidence:       {mean_conf_after:.3f}")
print(f"  Median confidence:     {median_conf_after:.3f}")
print(f"  Mean margin:           {mean_margin_after:.3f}")
print(f"  % ambiguous (margin<0.05): {pct_ambiguous_after:.1f}%")
print(f"  % HDBSCAN noise:       {pct_noise_after:.1f}%")

# Category distribution after
print(f"\nCategory distribution (after):")
cat_counts_after = plot_df_filtered['category'].value_counts()
for cat, count in cat_counts_after.items():
    pct = count / n_after * 100
    print(f"    {cat}: {count:,} ({pct:.1f}%)")

# =============================================================================
# SUMMARY COMPARISON TABLE
# =============================================================================
print(f"\n{'='*70}")
print("FILTERING IMPACT SUMMARY")
print(f"{'='*70}")

print(f"\n{'Metric':<30} {'Before':>12} {'After':>12} {'Change':>12}")
print("-" * 70)
print(f"{'Total files':<30} {n_before:>12,} {n_after:>12,} {n_after - n_before:>+12,}")
print(f"{'Total repositories':<30} {n_repos_before:>12,} {n_repos_after:>12,} {n_repos_after - n_repos_before:>+12,}")
print(f"{'Mean confidence':<30} {mean_conf_before:>12.3f} {mean_conf_after:>12.3f} {mean_conf_after - mean_conf_before:>+12.3f}")
print(f"{'Mean margin':<30} {mean_margin_before:>12.3f} {mean_margin_after:>12.3f} {mean_margin_after - mean_margin_before:>+12.3f}")
print(f"{'% ambiguous (margin<0.05)':<30} {pct_ambiguous_before:>11.1f}% {pct_ambiguous_after:>11.1f}% {pct_ambiguous_after - pct_ambiguous_before:>+11.1f}%")
print(f"{'% HDBSCAN noise':<30} {pct_noise_before:>11.1f}% {pct_noise_after:>11.1f}% {pct_noise_after - pct_noise_before:>+11.1f}%")

# Retention rate
retention_rate = n_after / n_before * 100
print(f"\n📊 Overall retention rate: {retention_rate:.1f}% ({n_after:,} / {n_before:,} files)")


In [ ]:
# Phase 6b.1: Before vs After UMAP Comparison
# Side-by-side visualization showing filtering impact

from plotly.subplots import make_subplots

print("Creating before/after UMAP comparison...")

# Sample for visualization (to avoid overcrowding)
sample_before = plot_df.sample(n=min(5000, len(plot_df)), random_state=42)
sample_after = plot_df_filtered.sample(n=min(5000, len(plot_df_filtered)), random_state=42)

fig_compare = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'BEFORE Filtering (n={len(plot_df):,})',
        f'AFTER Filtering (n={len(plot_df_filtered):,})'
    ),
    horizontal_spacing=0.08
)

# BEFORE - colored by category
for category in category_names:
    cat_data = sample_before[sample_before['category'] == category]
    color = category_colors.get(category, '#7f7f7f')
    
    fig_compare.add_trace(
        go.Scatter(
            x=cat_data['umap_x'],
            y=cat_data['umap_y'],
            mode='markers',
            name=category,
            marker=dict(size=5, color=color, opacity=0.6),
            showlegend=False,
        ),
        row=1, col=1
    )

# AFTER - colored by category
for category in category_names:
    cat_data = sample_after[sample_after['category'] == category]
    color = category_colors.get(category, '#7f7f7f')
    
    fig_compare.add_trace(
        go.Scatter(
            x=cat_data['umap_x'],
            y=cat_data['umap_y'],
            mode='markers',
            name=category,
            marker=dict(size=5, color=color, opacity=0.6),
            legendgroup=category,
            showlegend=True,
        ),
        row=1, col=2
    )

# Add centroids to both plots
for _, row in centroid_df.iterrows():
    category = row['category']
    color = category_colors.get(category, '#7f7f7f')
    
    # Before plot
    fig_compare.add_trace(
        go.Scatter(
            x=[row['umap_x']], y=[row['umap_y']],
            mode='markers',
            marker=dict(size=15, color=color, symbol='star', line=dict(width=1, color='black')),
            showlegend=False,
        ),
        row=1, col=1
    )
    
    # After plot
    fig_compare.add_trace(
        go.Scatter(
            x=[row['umap_x']], y=[row['umap_y']],
            mode='markers',
            marker=dict(size=15, color=color, symbol='star', line=dict(width=1, color='black')),
            showlegend=False,
        ),
        row=1, col=2
    )

fig_compare.update_layout(
    title=dict(
        text='Noise Filtering Impact: Before vs After<br><sup>Combined filter: Confidence ≥0.22, Margin ≥0.03, HDBSCAN Outlier ≤0.7</sup>',
        x=0.5
    ),
    height=500,
    width=1200,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

fig_compare.update_xaxes(title_text="UMAP Dimension 1", row=1, col=1)
fig_compare.update_xaxes(title_text="UMAP Dimension 1", row=1, col=2)
fig_compare.update_yaxes(title_text="UMAP Dimension 2", row=1, col=1)
fig_compare.update_yaxes(title_text="UMAP Dimension 2", row=1, col=2)

fig_compare.show()

# Print summary for slides
print("\n" + "=" * 70)
print("SUMMARY FOR SLIDES")
print("=" * 70)
print(f"""
| Metric | Before | After | Improvement |
|--------|--------|-------|-------------|
| Total files | {len(plot_df):,} | {len(plot_df_filtered):,} | {len(plot_df_filtered)/len(plot_df)*100:.1f}% retained |
| Repositories | {plot_df['repo_name'].nunique():,} | {plot_df_filtered['repo_name'].nunique():,} | {plot_df_filtered['repo_name'].nunique()/plot_df['repo_name'].nunique()*100:.1f}% retained |
| Mean confidence | {plot_df['confidence'].mean():.3f} | {plot_df_filtered['confidence'].mean():.3f} | +{plot_df_filtered['confidence'].mean() - plot_df['confidence'].mean():.3f} |
| HDBSCAN noise % | {(plot_df['hdbscan_label']==-1).mean()*100:.1f}% | {(plot_df_filtered['hdbscan_label']==-1).mean()*100:.1f}% | -{(plot_df['hdbscan_label']==-1).mean()*100 - (plot_df_filtered['hdbscan_label']==-1).mean()*100:.1f}% |
""")


In [ ]:
# Visualization 1b: FILTERED - Color by CATEGORY

# Compute filter statistics for title
filtered_files = len(plot_df_filtered)
dropped_files = len(plot_df) - filtered_files
drop_pct = dropped_files / len(plot_df) * 100

# Recompute centroid positions from filtered data
print("Recomputing centroid positions from filtered data...")

centroid_coords_filtered = []
for category in category_names:
    mask = (plot_df_filtered['category'] == category)
    if mask.any():
        centroid_x = plot_df_filtered.loc[mask, 'umap_x'].mean()
        centroid_y = plot_df_filtered.loc[mask, 'umap_y'].mean()
        n_files = mask.sum()
        print(f"  {category}: ({centroid_x:.2f}, {centroid_y:.2f}) from {n_files} files")
    else:
        # Fallback: use unfiltered centroid position
        centroid_x = centroid_df[centroid_df['category'] == category]['umap_x'].values[0]
        centroid_y = centroid_df[centroid_df['category'] == category]['umap_y'].values[0]
        print(f"  {category}: ({centroid_x:.2f}, {centroid_y:.2f}) - NO FILES, using original")
    centroid_coords_filtered.append({'category': category, 'umap_x': centroid_x, 'umap_y': centroid_y})

centroid_df_filtered = pd.DataFrame(centroid_coords_filtered)

# Create filtered visualization
fig1b = go.Figure()

# Add filtered file points grouped by category
for category in sorted(plot_df_filtered['category'].unique()):
    cat_data = plot_df_filtered[plot_df_filtered['category'] == category]
    color = category_colors.get(category, '#7f7f7f')
    
    fig1b.add_trace(go.Scatter(
        x=cat_data['umap_x'],
        y=cat_data['umap_y'],
        mode='markers',
        name=f"{category} ({len(cat_data)})",
        marker=dict(size=8, color=color, opacity=0.8, line=dict(width=1, color='white')),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>" +
            "Repo: %{customdata[1]}<br>" +
            "Path: %{customdata[2]}<br>" +
            "Category: %{customdata[3]} (conf: %{customdata[4]:.3f})<br>" +
            "2nd: %{customdata[5]}<br>" +
            "<extra></extra>"
        ),
        customdata=cat_data[['artifact_name', 'repo_name', 'artifact_path', 'category', 'confidence', 'second_category']].values,
        legendgroup=category,
    ))

# Add category centroids (recalculated from filtered data)
for _, row in centroid_df_filtered.iterrows():
    category = row['category']
    color = category_colors.get(category, '#7f7f7f')
    
    fig1b.add_trace(go.Scatter(
        x=[row['umap_x']],
        y=[row['umap_y']],
        mode='markers+text',
        marker=dict(size=20, color=color, symbol='star', line=dict(width=2, color='black')),
        text=[category.upper()],
        textposition='top center',
        textfont=dict(size=10, color='black'),
        hovertemplate=f"<b>{category.upper()} CENTROID</b><extra></extra>",
        legendgroup=category,
        showlegend=False,
    ))

fig1b.update_layout(
    title=f'FILTERED: All Repositories - Colored by Category<br><sup>{filtered_files:,} files retained ({100-drop_pct:.1f}%), {dropped_files:,} dropped ({drop_pct:.1f}%) | conf≥{FILTER_CONFIDENCE_MIN}, margin≥{FILTER_MARGIN_MIN}</sup>',
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    width=1200,
    height=800,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02, title="Categories"),
    hovermode='closest',
)

fig1b.show()

In [ ]:
# Visualization 1c: Show WHAT was filtered out (dropped points in grey)
fig1c = go.Figure()

# Get dropped files
plot_df_dropped = plot_df[~mask_combined].copy()

# Calculate margin for hover info
plot_df_dropped['margin'] = plot_df_dropped['confidence'] - plot_df_dropped['second_confidence']

# Add dropped files first (grey, semi-transparent background)
fig1c.add_trace(go.Scatter(
    x=plot_df_dropped['umap_x'],
    y=plot_df_dropped['umap_y'],
    mode='markers',
    name=f"Filtered out ({len(plot_df_dropped):,})",
    marker=dict(size=6, color='lightgray', opacity=0.4),
    hovertemplate=(
        "<b>DROPPED:</b> %{customdata[0]}<br>" +
        "Repo: %{customdata[1]}<br>" +
        "Category: %{customdata[2]} (conf: %{customdata[3]:.3f})<br>" +
        "Margin: %{customdata[4]:.3f}<br>" +
        "<extra></extra>"
    ),
    customdata=plot_df_dropped[['artifact_name', 'repo_name', 'category', 'confidence', 'margin']].values,
))

# Add retained files colored by category
for category in sorted(plot_df_filtered['category'].unique()):
    cat_data = plot_df_filtered[plot_df_filtered['category'] == category]
    color = category_colors.get(category, '#7f7f7f')
    
    fig1c.add_trace(go.Scatter(
        x=cat_data['umap_x'],
        y=cat_data['umap_y'],
        mode='markers',
        name=f"{category} ({len(cat_data)})",
        marker=dict(size=8, color=color, opacity=0.85, line=dict(width=1, color='white')),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>" +
            "Repo: %{customdata[1]}<br>" +
            "Category: %{customdata[2]} (conf: %{customdata[3]:.3f})<br>" +
            "<extra></extra>"
        ),
        customdata=cat_data[['artifact_name', 'repo_name', 'category', 'confidence']].values,
    ))

# Add centroids
for _, row in centroid_df_filtered.iterrows():
    category = row['category']
    color = category_colors.get(category, '#7f7f7f')
    
    fig1c.add_trace(go.Scatter(
        x=[row['umap_x']],
        y=[row['umap_y']],
        mode='markers+text',
        marker=dict(size=18, color=color, symbol='star', line=dict(width=2, color='black')),
        text=[category.upper()],
        textposition='top center',
        textfont=dict(size=9, color='black'),
        showlegend=False,
        hovertemplate=f"<b>{category.upper()} CENTROID</b><extra></extra>",
    ))

fig1c.update_layout(
    title=f'Before vs After Filtering: Grey = Dropped<br><sup>Retained: {filtered_files:,} ({100-drop_pct:.1f}%) | Dropped: {dropped_files:,} ({drop_pct:.1f}%)</sup>',
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    width=1200,
    height=800,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02),
    hovermode='closest',
)

fig1c.show()

In [ ]:
# Visualization 2: Color by REPOSITORY (Filtered Data)
fig2 = go.Figure()

# Add file points grouped by repository (using filtered data)
for repo in sorted(plot_df_filtered['repo_name'].unique()):
    repo_data = plot_df_filtered[plot_df_filtered['repo_name'] == repo]
    color = repo_colors.get(repo, '#7f7f7f')
    
    fig2.add_trace(go.Scatter(
        x=repo_data['umap_x'],
        y=repo_data['umap_y'],
        mode='markers',
        name=f"{repo} ({len(repo_data)})",
        marker=dict(size=8, color=color, opacity=0.7),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>" +
            "Repo: %{customdata[1]}<br>" +
            "Category: %{customdata[2]}<br>" +
            "<extra></extra>"
        ),
        customdata=repo_data[['artifact_name', 'repo_name', 'category']].values,
    ))

fig2.update_layout(
    title=f'Filtered Artifacts by Repository (n={len(plot_df_filtered):,} files)',
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    height=700,
    width=1000,
    showlegend=True,
)
fig2.show()


In [ ]:
# Visualization 3: Quality Distribution in Filtered Data
# Show confidence distribution of the filtered dataset

fig3 = make_subplots(rows=1, cols=2, subplot_titles=(
    'Confidence Distribution (Filtered)',
    'Category Distribution (Filtered)'
))

# Confidence histogram
fig3.add_trace(
    go.Histogram(x=plot_df_filtered['confidence'], nbinsx=30,
                 marker_color='steelblue', opacity=0.7),
    row=1, col=1
)

# Category bar chart
cat_counts = plot_df_filtered['category'].value_counts()
fig3.add_trace(
    go.Bar(x=cat_counts.index, y=cat_counts.values,
           marker_color=[category_colors.get(c, '#7f7f7f') for c in cat_counts.index]),
    row=1, col=2
)

fig3.update_layout(
    title=f'Filtered Dataset Quality (n={len(plot_df_filtered):,} files)',
    height=400,
    width=1100,
    showlegend=False
)
fig3.update_xaxes(title_text="Confidence Score", row=1, col=1)
fig3.update_xaxes(title_text="Category", row=1, col=2)
fig3.update_yaxes(title_text="Count", row=1, col=1)
fig3.update_yaxes(title_text="Count", row=1, col=2)

fig3.show()

print(f"Filtered dataset statistics:")
print(f"  Mean confidence: {plot_df_filtered['confidence'].mean():.3f}")
print(f"  Median confidence: {plot_df_filtered['confidence'].median():.3f}")
print(f"  Min confidence: {plot_df_filtered['confidence'].min():.3f}")


In [ ]:
# Visualization 4: Category separation heatmap
print("=" * 60)
print("CATEGORY SEPARATION ANALYSIS")
print("=" * 60)

# Compute pairwise cosine similarities between category centroids
centroid_similarities = cosine_similarity(category_matrix)

fig4 = go.Figure(data=go.Heatmap(
    z=centroid_similarities,
    x=category_names,
    y=category_names,
    colorscale='RdYlBu_r',
    text=np.round(centroid_similarities, 2),
    texttemplate='%{text}',
    textfont={"size": 10},
    hovertemplate='%{x} vs %{y}<br>Similarity: %{z:.3f}<extra></extra>',
))

fig4.update_layout(
    title='Category Centroid Similarities<br><sup>Lower values = better separated categories</sup>',
    width=700,
    height=600,
)

fig4.show()

# Print the most similar category pairs
print("\nMost similar category pairs (potential overlap):")
for i in range(len(category_names)):
    for j in range(i+1, len(category_names)):
        sim = centroid_similarities[i, j]
        if sim > 0.5:
            print(f"  {category_names[i]} <-> {category_names[j]}: {sim:.3f}")

## Phase 7: Export Results

In [ ]:
# Export classification results
print("=" * 70)
print("EXPORTING RESULTS")
print("=" * 70)

# Use data/ folder for analytical outputs (separate from raw embeddings in output/)
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

# Build export DataFrame with all artifacts
export_df = plot_df[[
    'global_file_id', 'repo_name', 'artifact_name', 'artifact_path', 'tool_name',
    'category', 'confidence', 'second_category', 'second_confidence',
    'umap_x', 'umap_y', 'centroid_distance', 'hdbscan_label', 'hdbscan_outlier_score', 'local_density_dist'
]].copy()

# Add margin and filter flag
export_df['margin'] = export_df['confidence'] - export_df['second_confidence']
export_df['is_filtered'] = mask_combined.values

# Export to CSV
output_path = DATA_DIR / "artifacts_all.csv"
export_df.to_csv(output_path, index=False)

n_total = len(export_df)
n_filtered = export_df['is_filtered'].sum()
print(f"\nExported {n_total:,} artifacts to: {output_path}")
print(f"  - Filtered (is_filtered=True): {n_filtered:,} ({n_filtered/n_total*100:.1f}%)")
print(f"  - Dropped (is_filtered=False): {n_total - n_filtered:,} ({(n_total-n_filtered)/n_total*100:.1f}%)")

print(f"\nExport complete!")

## Summary

In [ ]:
print("=" * 70)
print("FINAL CLASSIFICATION SUMMARY (FILTERED)")
print("=" * 70)

print(f"\n📊 FILTERED DATASET STATISTICS")
print(f"  Files: {len(plot_df_filtered):,} (from {len(plot_df):,} total, {len(plot_df_filtered)/len(plot_df)*100:.1f}% retained)")
print(f"  Repositories: {plot_df_filtered['repo_name'].nunique():,}")
print(f"  Categories: {plot_df_filtered['category'].nunique()}")

print(f"\n📈 QUALITY METRICS")
print(f"  Mean confidence: {plot_df_filtered['confidence'].mean():.3f}")
print(f"  Median confidence: {plot_df_filtered['confidence'].median():.3f}")
print(f"  Mean margin: {(plot_df_filtered['confidence'] - plot_df_filtered['second_confidence']).mean():.3f}")

print(f"\n📁 CATEGORY DISTRIBUTION")
for cat in category_names:
    count = (plot_df_filtered['category'] == cat).sum()
    pct = count / len(plot_df_filtered) * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")

print(f"\n🔧 TOOL DISTRIBUTION")
tool_counts = plot_df_filtered['tool_name'].value_counts()
for tool, count in tool_counts.head(10).items():
    pct = count / len(plot_df_filtered) * 100
    print(f"  {tool}: {count:,} ({pct:.1f}%)")

print(f"\n✅ Analysis complete!")


## Phase 8: Save Filtered Data for Downstream Analysis

Save filtered embeddings and metadata to `data/` folder for use in other notebooks (e.g., temporal analysis).

In [ ]:
# Save filtered embeddings and classification model for downstream analysis

# Filter embeddings using the combined mask from Phase 6b
embeddings_filtered = embeddings[mask_combined.values]

# 1. Save filtered embeddings with alignment info
filtered_embeddings_data = {
    'embeddings': embeddings_filtered,
    'global_file_ids': plot_df_filtered['global_file_id'].tolist(),
    'model': model_name,
}

filtered_embeddings_path = DATA_DIR / 'embeddings_filtered.pkl'
with open(filtered_embeddings_path, 'wb') as f:
    pickle.dump(filtered_embeddings_data, f)

print(f"Saved filtered embeddings to: {filtered_embeddings_path}")
print(f"  - Shape: {embeddings_filtered.shape}")

# 2. Save classification model (centroids + all filter params) for future inference
classification_model = {
    # Category centroids for nearest-centroid classification
    'category_embeddings': category_embeddings,  # dict: category_name -> 768-dim vector
    'category_names': category_names,
    'category_templates': CATEGORY_TEMPLATES,
    
    # Embedding model used
    'embedding_model': model_name,
    
    # All filter parameters (for reference, even if only some are used)
    'filter_params': {
        'confidence_min': FILTER_CONFIDENCE_MIN,
        'margin_min': FILTER_MARGIN_MIN,
        'hdbscan_outlier_max': FILTER_HDBSCAN_OUTLIER_MAX,
    },
}

classification_model_path = DATA_DIR / 'classification_model.pkl'
with open(classification_model_path, 'wb') as f:
    pickle.dump(classification_model, f)

print(f"\nSaved classification model to: {classification_model_path}")
print(f"  - Categories: {category_names}")
print(f"  - Embedding model: {model_name}")
print(f"  - Filter params: conf >= {FILTER_CONFIDENCE_MIN}, margin >= {FILTER_MARGIN_MIN}, hdbscan <= {FILTER_HDBSCAN_OUTLIER_MAX}")

print(f"\n✅ All data saved to {DATA_DIR}/")